In [1]:
import pandas as pd 

# df_ground_truth = pd.read_csv("data/ground_truth.csv")
# df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
df_ground_truth.head()

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


In [3]:
from src.ingest import load_faq_data, build_index

documents = load_faq_data() 

documents_llm = [] 

for doc in documents: 
    if doc["course"] == "llm-zoomcamp": 
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [5]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [6]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [7]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


"Relevance matrix": Generated Documents (or Questions) by Retrieved Documents

In [8]:
relevance = [] 
for d in results: 
    relevance.append(int(d["id"] == doc_id))

relevance 

[1, 0, 0, 0, 0]

In [9]:
def compute_relevance_text(q): 
    doc_id = q["document"] 
    results = text_search(query=q["question"]) 

    relevance = [] 
    for d in results: 
        relevance.append(int(d["id"] == doc_id)) 
    
    return relevance 

In [10]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [11]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

If the vector was something like $[0,0,1,0,0]$ then the relevant document would be retrieved in the third position. 

Now compute for the first 15 documents: 

In [12]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [13]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [14]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Generic function (can use any search method, also for vector/hybrid search): 

In [15]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Let's now run it with all documents: 

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

**Metrics & Search Evaluation**

**Hit Rate**. In how many cases were we able to retrieve the document we wanted? 

In [18]:
import numpy as np
np.array(relevance_total).sum(axis=1).sum() / len(relevance_total)

np.float64(0.6734177215189874)

In [19]:
def hit_rate(relevance): 
    cnt = 0 

    for line in relevance: 
        if 1 in line: 
            cnt += 1 

    return cnt / len(relevance)

In [65]:
print(hit_rate(relevance_total_text))  # -> on the sample 
print(hit_rate(relevance_total)) # -> on all documents

0.8666666666666667
0.6734177215189874


**Mean Reciprocal Rank** (MRR). Average score based on document rank. MRR <= HIT RATE (Hit rate is the upper bound of MRR)

In [22]:
total_score = 0.0 

for line in relevance_total_text: 
    for rank in range(len(line)): 
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score 
            break 

total_score / len(relevance_total_text)

0.7333333333333333

In [26]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


Put it all together: 

In [27]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [28]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6734177215189874, 'mrr': 0.5895780590717299}

**Search Parameter Tuning**. 

In [29]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [30]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6835443037974683, 'mrr': 0.5962447257383967}

In [31]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [32]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.7139240506329114, 'mrr': 0.620084388185654}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.7139240506329114, 'mrr': 0.6206751054852321}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6734177215189874, 'mrr': 0.5895780590717299}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.660759493670886, 'mrr': 0.5727848101265822}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6455696202531646, 'mrr': 0.560168776371308}


Boost = 1.0 seems to be the best configuration

LEt's now try tuning all hyperparameters with grid search:  

In [33]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [34]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

Sort by MRR: 

In [35]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.767089,0.675865
19,2.0,4.0,0.2,0.767089,0.675865
35,5.0,10.0,0.5,0.767089,0.675865
34,5.0,10.0,0.2,0.767089,0.675654
18,2.0,4.0,0.1,0.767089,0.675654
33,5.0,10.0,0.1,0.764557,0.675021
4,1.0,2.0,0.2,0.764557,0.671646
20,2.0,4.0,0.5,0.759494,0.671224
7,1.0,4.0,0.2,0.759494,0.669958
6,1.0,4.0,0.1,0.759494,0.667764


In reality, you can optimize with some smart optimizer such as `hyperopt` or `optuna`  